In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)

df = pd.read_csv('../data/delhi_ncr_aqi_dataset.csv')
print(f"Dataset loaded — Shape: {df.shape}")
df.head()

Dataset loaded — Shape: (201664, 25)


,datetime,date,year,month,day,hour,day_of_week,is_weekend,season,city,...,no2,so2,co,o3,temperature,humidity,wind_speed,visibility,aqi,aqi_category
0,01-01-2020 06:00,01-01-2020,2020,1,1,6,Wednesday,0,winter,Delhi,...,119.6,47.7,5.19,12.3,9.4,100,3.6,1.2,500,Severe
1,01-01-2020 12:00,01-01-2020,2020,1,1,12,Wednesday,0,winter,Delhi,...,117.9,39.3,4.32,15.8,20.6,50,5.9,1.4,500,Severe
2,01-01-2020 18:00,01-01-2020,2020,1,1,18,Wednesday,0,winter,Delhi,...,150.1,36.3,7.13,14.3,12.4,56,4.5,1.1,500,Severe
3,01-01-2020 23:00,01-01-2020,2020,1,1,23,Wednesday,0,winter,Delhi,...,142.0,30.3,4.90,13.2,14.4,48,5.8,1.4,500,Severe
4,01-01-2020 06:00,01-01-2020,2020,1,1,6,Wednesday,0,winter,Delhi,...,138.4,41.5,7.56,15.4,6.8,100,2.8,0.4,500,Severe


In [3]:
# Columns to drop:
# - datetime, date     : redundant — year/month/day/hour already extracted
# - latitude, longitude: station name already captures location
# - aqi_category       : derived from AQI — would cause data leakage

cols_to_drop = ['datetime', 'date', 'latitude', 'longitude', 'aqi_category']
df = df.drop(columns=cols_to_drop)

print(f"Columns dropped: {cols_to_drop}")
print(f"\nRemaining columns ({len(df.columns)}):")
print(df.columns.tolist())
print(f"\nShape after dropping: {df.shape}")

Columns dropped: ['datetime', 'date', 'latitude', 'longitude', 'aqi_category']

Remaining columns (20):
['year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend', 'season', 'city', 'station', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'wind_speed', 'visibility', 'aqi']

Shape after dropping: (201664, 20)


In [4]:
pollutant_cols = ['pm25', 'pm10', 'no2', 'so2', 'co', 'o3']

print("=== Outlier Analysis (IQR Method) ===")
print(f"{'Feature':<12} {'Q1':>8} {'Q3':>8} {'IQR':>8} "
      f"{'Lower':>8} {'Upper':>8} {'Outliers':>10} {'%':>6}")
print("-" * 70)

for col in pollutant_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = outliers / len(df) * 100
    print(f"{col:<12} {Q1:>8.1f} {Q3:>8.1f} {IQR:>8.1f} "
          f"{lower:>8.1f} {upper:>8.1f} {outliers:>10,} {pct:>6.2f}%")

=== Outlier Analysis (IQR Method) ===
Feature            Q1       Q3      IQR    Lower    Upper   Outliers      %
----------------------------------------------------------------------
pm25             55.3    254.7    199.4   -243.8    553.8     12,877   6.39%
pm10            104.1    481.2    377.1   -461.6   1046.9     12,980   6.44%
no2              19.9     94.0     74.1    -91.2    205.1     13,163   6.53%
so2               4.5     20.9     16.4    -20.1     45.5     13,776   6.83%
co                0.9      4.1      3.2     -4.0      9.0     12,961   6.43%
o3               18.0     31.8     13.8     -2.7     52.5     12,696   6.30%


In [5]:
print("=== Outlier Treatment — Capping at 99th Percentile ===\n")

for col in pollutant_cols:
    before_max = df[col].max()
    cap_value = df[col].quantile(0.99)
    outliers_count = (df[col] > cap_value).sum()
    df[col] = df[col].clip(upper=cap_value)
    after_max = df[col].max()
    print(f"{col:<12}: capped at {cap_value:.2f} "
          f"— {outliers_count:,} values clipped "
          f"| max before: {before_max:.2f} "
          f"→ max after: {after_max:.2f}")

print(f"\nDataset shape after treatment: {df.shape}")
print("\nUpdated statistics after capping:")
df[pollutant_cols].describe().round(2)

=== Outlier Treatment — Capping at 99th Percentile ===

pm25        : capped at 900.00 — 0 values clipped | max before: 900.00 → max after: 900.00
pm10        : capped at 1770.00 — 2,016 values clipped | max before: 1979.70 → max after: 1770.00
no2         : capped at 358.54 — 2,017 values clipped | max before: 593.50 → max after: 358.54
so2         : capped at 85.10 — 2,014 values clipped | max before: 121.60 → max after: 85.10
co          : capped at 15.57 — 2,005 values clipped | max before: 22.67 → max after: 15.57
o3          : capped at 79.50 — 2,007 values clipped | max before: 84.00 → max after: 79.50

Dataset shape after treatment: (201664, 20)

Updated statistics after capping:


,pm25,pm10,no2,so2,co,o3
count,201664.00,201664.00,201664.00,201664.00,201664.00,201664.00
mean,183.42,347.57,69.15,15.91,3.01,27.16
std,193.14,366.00,73.15,16.67,3.18,13.48
min,15.00,24.00,8.00,4.00,0.30,12.00
25%,55.30,104.10,19.90,4.50,0.87,18.00
50%,99.50,189.80,38.30,8.70,1.69,23.30
75%,254.70,481.20,94.00,20.90,4.12,31.80
max,900.00,1770.00,358.54,85.10,15.57,79.50


In [6]:
# Check current string columns
string_cols = df.select_dtypes(include='object').columns.tolist()
print(f"String columns to encode: {string_cols}")
print()
for col in string_cols:
    print(f"{col}: {df[col].unique().tolist()}")

String columns to encode: ['day_of_week', 'season', 'city', 'station']

day_of_week: ['Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday', 'Monday', 'Tuesday']


season: ['winter', 'summer', 'monsoon', 'post_monsoon']
city: ['Delhi', 'Noida', 'Gurugram', 'Faridabad', 'Ghaziabad']
station: ['Anand Vihar, Delhi', 'Jahangirpuri, Delhi', 'Wazirpur, Delhi', 'Bawana, Delhi', 'ITO, Delhi', 'Punjabi Bagh, Delhi', 'Okhla Phase 2, Delhi', 'Shadipur, Delhi', 'Rohini, Delhi', 'RK Puram, Delhi', 'Siri Fort, Delhi', 'Dwarka Sec 8, Delhi', 'NSIT Dwarka, Delhi', 'Mandir Marg, Delhi', 'Noida Sec 62', 'Noida Sec 125', 'Greater Noida', 'Gurugram Vikas Sadan', 'Gurugram Sec 51', 'Faridabad Sec 16A', 'Faridabad New Town', 'Ghaziabad Vasundhara', 'Ghaziabad Loni']


In [7]:
from sklearn.preprocessing import LabelEncoder

# 1. Season — Ordinal encoding (order matters)
season_map = {'monsoon': 1, 'summer': 2, 'post_monsoon': 3, 'winter': 4}
df['season'] = df['season'].map(season_map)
print(f"Season encoded: {sorted(df['season'].unique())}")

# 2. Day of week — Ordinal encoding
day_map = {
    'Monday': 1, 'Tuesday': 2, 'Wednesday': 3,
    'Thursday': 4, 'Friday': 5, 'Saturday': 6, 'Sunday': 7
}
df['day_of_week'] = df['day_of_week'].map(day_map)
print(f"Day of week encoded: {sorted(df['day_of_week'].unique())}")

# 3. City & Station — Label encoding
le_city = LabelEncoder()
le_station = LabelEncoder()

df['city'] = le_city.fit_transform(df['city'])
df['station'] = le_station.fit_transform(df['station'])

print(f"City encoded: {sorted(df['city'].unique())}")
print(f"Station encoded: {sorted(df['station'].unique())}")

# Verify no string columns remain
remaining_strings = df.select_dtypes(include='object').columns.tolist()
print(f"\nRemaining string columns: {remaining_strings}")
print(f"All columns are now numeric: {len(remaining_strings) == 0}")

Season encoded: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Day of week encoded: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
City encoded: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Station encoded: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]

Remaining string columns: []
All columns are now numeric: True


In [8]:
# Sort by station and time before creating lag features
# This ensures lag features don't bleed across different stations
df = df.sort_values(['station', 'year', 'month', 'day', 'hour'])
df = df.reset_index(drop=True)
print("Data sorted by station and time ✓")
print(df[['station', 'year', 'month', 'day', 'hour', 'aqi']].head(10))

Data sorted by station and time ✓
   station  year  month  day  hour  aqi
0        0  2020      1    1     6  500
1        0  2020      1    1    12  500
2        0  2020      1    1    18  500
3        0  2020      1    1    23  500
4        0  2020      1    2     6  500
5        0  2020      1    2    12  500
6        0  2020      1    2    18  500
7        0  2020      1    2    23  500
8        0  2020      1    3     6  500
9        0  2020      1    3    12  500


In [9]:
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop(columns=['aqi'])
y = df['aqi']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"\nFeature columns ({len(X.columns)}):")
for i, col in enumerate(X.columns.tolist(), 1):
    print(f"  {i:>2}. {col}")

Features shape : (201664, 19)
Target shape   : (201664,)

Feature columns (19):
   1. year
   2. month
   3. day
   4. hour
   5. day_of_week
   6. is_weekend
   7. season
   8. city
   9. station
  10. pm25
  11. pm10
  12. no2
  13. so2
  14. co
  15. o3
  16. temperature
  17. humidity
  18. wind_speed
  19. visibility


In [10]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Features scaled successfully!")
print(f"\nBefore scaling — PM2.5 stats:")
print(f"  Mean : {X['pm25'].mean():.2f}")
print(f"  Std  : {X['pm25'].std():.2f}")
print(f"  Min  : {X['pm25'].min():.2f}")
print(f"  Max  : {X['pm25'].max():.2f}")
print(f"\nAfter scaling — PM2.5 stats:")
print(f"  Mean : {X_scaled['pm25'].mean():.4f}  (should be ~0)")
print(f"  Std  : {X_scaled['pm25'].std():.4f}   (should be ~1)")
print(f"  Min  : {X_scaled['pm25'].min():.4f}")
print(f"  Max  : {X_scaled['pm25'].max():.4f}")

Features scaled successfully!

Before scaling — PM2.5 stats:
  Mean : 183.42
  Std  : 193.14
  Min  : 15.00
  Max  : 900.00

After scaling — PM2.5 stats:
  Mean : -0.0000  (should be ~0)
  Std  : 1.0000   (should be ~1)
  Min  : -0.8720
  Max  : 3.7102


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42
)

print("=== Train-Test Split ===")
print(f"Total samples  : {len(X_scaled):,}")
print(f"Training set   : {len(X_train):,} ({len(X_train)/len(X_scaled)*100:.1f}%)")
print(f"Testing set    : {len(X_test):,}  ({len(X_test)/len(X_scaled)*100:.1f}%)")
print(f"Features       : {X_train.shape[1]}")
print(f"\nTarget distribution — Training set:")
print(f"  Mean AQI : {y_train.mean():.2f}")
print(f"  Min AQI  : {y_train.min():.2f}")
print(f"  Max AQI  : {y_train.max():.2f}")
print(f"\nTarget distribution — Testing set:")
print(f"  Mean AQI : {y_test.mean():.2f}")
print(f"  Min AQI  : {y_test.min():.2f}")
print(f"  Max AQI  : {y_test.max():.2f}")

=== Train-Test Split ===
Total samples  : 201,664
Training set   : 161,331 (80.0%)
Testing set    : 40,333  (20.0%)
Features       : 19

Target distribution — Training set:
  Mean AQI : 265.88
  Min AQI  : 25.00
  Max AQI  : 500.00

Target distribution — Testing set:
  Mean AQI : 265.64
  Min AQI  : 25.00
  Max AQI  : 500.00


In [12]:
import os
import pickle

# Create necessary folders
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save train/test splits
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save scaler for Flask app
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save encoders for Flask app
with open('../models/le_city.pkl', 'wb') as f:
    pickle.dump(le_city, f)

with open('../models/le_station.pkl', 'wb') as f:
    pickle.dump(le_station, f)

# Verify all files saved
print("=== Files Saved Successfully ===")
for path in [
    '../data/processed/X_train.csv',
    '../data/processed/X_test.csv',
    '../data/processed/y_train.csv',
    '../data/processed/y_test.csv',
    '../models/scaler.pkl',
    '../models/le_city.pkl',
    '../models/le_station.pkl'
]:
    size = os.path.getsize(path)
    print(f"  ✓ {path:<40} ({size/1024:.1f} KB)")

=== Files Saved Successfully ===
  ✓ ../data/processed/X_train.csv            (58831.5 KB)
  ✓ ../data/processed/X_test.csv             (14708.2 KB)
  ✓ ../data/processed/y_train.csv            (751.2 KB)
  ✓ ../data/processed/y_test.csv             (187.8 KB)
  ✓ ../models/scaler.pkl                     (1.1 KB)
  ✓ ../models/le_city.pkl                    (0.3 KB)
  ✓ ../models/le_station.pkl                 (0.7 KB)


In [18]:
print("=" * 50)
print("      PREPROCESSING SUMMARY")
print("=" * 50)
print(f"  Original Shape     : (201,664 x 25)")
print(f"  Final Shape        : (201,664 x 20)")
print(f"  Columns Dropped    : 5")
print(f"                       datetime, date,")
print(f"                       latitude, longitude,")
print(f"                       aqi_category")
print("-" * 50)
print(f"  Outliers Treated   : Capped at 99th percentile")
print(f"                       pm10  — 2,016 values")
print(f"                       no2   — 2,017 values")
print(f"                       so2   — 2,014 values")
print(f"                       co    — 2,005 values")
print(f"                       o3    — 2,007 values")
print("-" * 50)
print(f"  Encoded Columns    : season, day_of_week,")
print(f"                       city, station")
print("-" * 50)
print(f"  Scaling            : StandardScaler applied")
print(f"  Training Samples   : 161,331 (80%)")
print(f"  Testing Samples    :  40,333 (20%)")
print(f"  Total Features     : 19")
print("=" * 50)
print("  Preprocessing Complete! Ready for Modeling.")
print("=" * 50)

      PREPROCESSING SUMMARY
  Original Shape     : (201,664 x 25)
  Final Shape        : (201,664 x 20)
  Columns Dropped    : 5
                       datetime, date,
                       latitude, longitude,
                       aqi_category
--------------------------------------------------
  Outliers Treated   : Capped at 99th percentile
                       pm10  — 2,016 values
                       no2   — 2,017 values
                       so2   — 2,014 values
                       co    — 2,005 values
                       o3    — 2,007 values
--------------------------------------------------
  Encoded Columns    : season, day_of_week,
                       city, station
--------------------------------------------------
  Scaling            : StandardScaler applied
  Training Samples   : 161,331 (80%)
  Testing Samples    :  40,333 (20%)
  Total Features     : 19
  Preprocessing Complete! Ready for Modeling.


In [14]:
print("=== AQI = 500 Analysis ===")
cap_count = (df['aqi'] == 500).sum()
cap_pct = cap_count / len(df) * 100
print(f"Rows with AQI = 500  : {cap_count:,}")
print(f"Percentage           : {cap_pct:.2f}%")
print(f"\nAQI value distribution (top 10 most frequent):")
print(df['aqi'].value_counts().head(10))

=== AQI = 500 Analysis ===
Rows with AQI = 500  : 45,355
Percentage           : 22.49%

AQI value distribution (top 10 most frequent):
aqi
500    45355
101     1043
103      947
102      930
104      926
105      918
109      894
100      866
108      860
106      852
Name: count, dtype: int64
